# 🔥 Heatmaps Tactiques Football
**Modèle : RT-DETR** | Analyse des positions joueurs + trajectoire ballon

---
### Outputs générés
| Fichier | Contenu |
|---|---|
| `heatmap_globale.png` | Zones de chaleur tous joueurs confondus |
| `heatmap_ballon.png` | Trajectoire & fréquence de passage du ballon |
| `heatmaps_individuelles.png` | Top 6 joueurs les plus actifs |
| `positions_finales.png` | Dernières positions connues de chaque joueur |

## ⚙️ 1. Installation

In [ ]:
!pip install ultralytics supervision matplotlib scipy

## 📦 2. Imports & Config

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict
from scipy.spatial import Voronoi

# ─── CONFIG ───────────────────────────────────────────
TERRAIN_W  = 1050
TERRAIN_H  = 680
VIDEO_PATH = "match.mp4"   # ⚠️ chemin vers ta vidéo
CONF       = 0.5
FRAME_SKIP = 3             # analyser 1 frame sur N (plus grand = plus rapide)
# ──────────────────────────────────────────────────────

print("✅ Imports OK")

## 🤖 3. Chargement du modèle RT-DETR

In [ ]:
model   = YOLO("rtdetr-l.pt")
tracker = sv.ByteTrack()
print("✅ RT-DETR chargé !")

## 📌 4. Sélection des 4 coins du terrain

> **Instructions :**
> 1. Une fenêtre va s'ouvrir avec la première frame
> 2. **Clique sur 4 coins** dans cet ordre : ↖️ → ↗️ → ↘️ → ↙️
> 3. Appuie sur **n'importe quelle touche** pour valider

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
cap.release()

if not ret:
    raise FileNotFoundError(f"❌ Impossible d'ouvrir : {VIDEO_PATH}")

src_points = []
frame_copy = first_frame.copy()

def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(src_points) < 4:
        src_points.append([x, y])
        cv2.circle(frame_copy, (x, y), 8, (0, 255, 0), -1)
        cv2.putText(frame_copy, str(len(src_points)), (x + 10, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
        cv2.imshow("📌 Selectionne 4 coins", frame_copy)
        if len(src_points) == 4:
            print("✅ 4 points sélectionnés !")

cv2.imshow("📌 Selectionne 4 coins", frame_copy)
cv2.setMouseCallback("📌 Selectionne 4 coins", click_event)
cv2.waitKey(0)
cv2.destroyAllWindows()
print(f"Points : {src_points}")

## 🔢 5. Calcul de la matrice d'homographie

In [ ]:
dst_points = np.float32([
    [0, 0], [TERRAIN_W, 0], [TERRAIN_W, TERRAIN_H], [0, TERRAIN_H]
])
H, _ = cv2.findHomography(np.float32(src_points), dst_points)
print("✅ Matrice H calculée")

def project_point(pt, H):
    p = np.float32([[pt]])
    proj = cv2.perspectiveTransform(p, H)
    return int(proj[0][0][0]), int(proj[0][0][1])

## 📹 6. Collecte des positions (analyse vidéo complète)

In [ ]:
positions      = defaultdict(list)   # joueurs  : {tracker_id: [(x,y), ...]}
ball_positions = []                  # ballon   : [(x,y), ...]

cap      = cv2.VideoCapture(VIDEO_PATH)
total    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_id = 0

print(f"📹 Analyse de {total} frames (1 sur {FRAME_SKIP})...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    if frame_id % FRAME_SKIP != 0:
        continue

    # ✅ RT-DETR
    results    = model(frame, conf=CONF)[0]
    detections = sv.Detections.from_ultralytics(results)
    mask       = np.isin(detections.class_id, [0, 32])
    detections = detections[mask]
    detections = tracker.update_with_detections(detections)

    for box, tid, cls_id in zip(detections.xyxy, detections.tracker_id, detections.class_id):
        foot_x = int((box[0] + box[2]) / 2)
        foot_y = int(box[3])
        tx, ty = project_point((foot_x, foot_y), H)

        if 0 <= tx < TERRAIN_W and 0 <= ty < TERRAIN_H:
            if cls_id == 32:
                ball_positions.append((tx, ty))
            else:
                positions[tid].append((tx, ty))

    if frame_id % 90 == 0:
        pct = round(frame_id / total * 100)
        print(f"  [{pct}%] Frame {frame_id}/{total} — {len(positions)} joueurs | {len(ball_positions)} pos. ballon")

cap.release()
total_pos = sum(len(v) for v in positions.values())
print(f"\n✅ Terminé — {total_pos} positions joueurs | {len(ball_positions)} positions ballon")

## 🎨 7. Fonction terrain matplotlib

In [ ]:
def draw_pitch_matplotlib(ax, w=TERRAIN_W, h=TERRAIN_H):
    """Terrain style tableau tactique pour matplotlib."""
    ax.set_facecolor("#f5f5f0")
    ax.set_xlim(0, w)
    ax.set_ylim(0, h)
    ax.invert_yaxis()
    lc, lw = "#222222", 2

    ax.add_patch(patches.Rectangle((10, 10), w - 20, h - 20,
                                    fill=False, edgecolor=lc, linewidth=lw))
    ax.plot([10, w - 10], [h // 2, h // 2], color=lc, lw=lw)
    ax.add_patch(patches.Circle((w // 2, h // 2), h * 0.14,
                                 fill=False, edgecolor=lc, lw=lw))
    ax.plot(w // 2, h // 2, "o", color=lc, ms=5)

    # Surfaces
    ax.add_patch(patches.Rectangle((w*0.21, 10),         w*0.58, h*0.147, fill=False, edgecolor=lc, lw=lw))
    ax.add_patch(patches.Rectangle((w*0.21, h*0.843),    w*0.58, h*0.147, fill=False, edgecolor=lc, lw=lw))

    # Petits rectangles
    ax.add_patch(patches.Rectangle((w*0.37, 10),         w*0.26, h*0.057, fill=False, edgecolor=lc, lw=lw))
    ax.add_patch(patches.Rectangle((w*0.37, h*0.943),    w*0.26, h*0.057, fill=False, edgecolor=lc, lw=lw))

    # Arcs
    ax.add_patch(patches.Arc((w//2, h*0.157), h*0.2, h*0.2, angle=0, theta1=30,  theta2=150, edgecolor=lc, lw=lw))
    ax.add_patch(patches.Arc((w//2, h*0.843), h*0.2, h*0.2, angle=0, theta1=210, theta2=330, edgecolor=lc, lw=lw))

    # Points penalty + cages
    ax.plot(w // 2, h * 0.11, "o", color=lc, ms=5)
    ax.plot(w // 2, h * 0.89, "o", color=lc, ms=5)
    ax.add_patch(patches.Rectangle((w*0.44, 10),      w*0.12, h*0.02, fill=False, edgecolor=lc, lw=lw))
    ax.add_patch(patches.Rectangle((w*0.44, h*0.98),  w*0.12, h*0.02, fill=False, edgecolor=lc, lw=lw))

    ax.axis("off")

print("✅ Fonction terrain prête")

## 🔥 8. Heatmap Globale — Tous les joueurs

In [ ]:
all_x = [x for pts in positions.values() for x, y in pts]
all_y = [y for pts in positions.values() for x, y in pts]

fig, ax = plt.subplots(figsize=(8, 10))
draw_pitch_matplotlib(ax)
hb = ax.hexbin(all_x, all_y, gridsize=30, cmap="hot", alpha=0.65,
               extent=[0, TERRAIN_W, 0, TERRAIN_H])
plt.colorbar(hb, ax=ax, label="Densité de présence")
ax.set_title("🔥 Heatmap Globale — Tous les joueurs", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig("heatmap_globale.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 heatmap_globale.png")

## ⚽ 9. Heatmap Trajectoire du Ballon

In [ ]:
if ball_positions:
    bx = [x for x, y in ball_positions]
    by = [y for x, y in ball_positions]

    fig, ax = plt.subplots(figsize=(8, 10))
    draw_pitch_matplotlib(ax)
    hb = ax.hexbin(bx, by, gridsize=25, cmap="YlOrRd", alpha=0.7,
                   extent=[0, TERRAIN_W, 0, TERRAIN_H])
    plt.colorbar(hb, ax=ax, label="Fréquence de passage")
    ax.set_title("⚽ Heatmap Trajectoire du Ballon", fontsize=14, pad=15)
    plt.tight_layout()
    plt.savefig("heatmap_ballon.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("💾 heatmap_ballon.png")
else:
    print("⚠️ Pas assez de positions ballon — essaie de baisser CONF à 0.35")

## 👤 10. Heatmaps Individuelles — Top 6 joueurs

In [ ]:
top_players = sorted(positions.keys(), key=lambda tid: len(positions[tid]), reverse=True)[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 14))
axes = axes.flatten()

for i, tid in enumerate(top_players):
    ax = axes[i]
    draw_pitch_matplotlib(ax)
    xs = [x for x, y in positions[tid]]
    ys = [y for x, y in positions[tid]]
    hb = ax.hexbin(xs, ys, gridsize=20, cmap="RdYlGn_r", alpha=0.7,
                   extent=[0, TERRAIN_W, 0, TERRAIN_H])
    plt.colorbar(hb, ax=ax)
    ax.set_title(f"Joueur #{tid} — {len(positions[tid])} positions", fontsize=11)

plt.suptitle("Heatmaps Individuelles — Top 6 joueurs les plus actifs", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig("heatmaps_individuelles.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 heatmaps_individuelles.png")

## 📍 11. Carte des dernières positions (snapshot tactique)

In [ ]:
last_positions = {
    tid: pts[-1] for tid, pts in positions.items() if len(pts) > 0
}

if len(last_positions) >= 4:
    fig, ax = plt.subplots(figsize=(8, 10))
    draw_pitch_matplotlib(ax)

    colors = plt.cm.tab20.colors
    for i, (tid, (px, py)) in enumerate(last_positions.items()):
        color = colors[i % len(colors)]
        ax.plot(px, py, "o", color=color, ms=12)
        ax.annotate(f"#{tid}", (px, py), textcoords="offset points",
                    xytext=(8, 4), fontsize=8, color=color, fontweight="bold")

    ax.set_title("📍 Snapshot Tactique — Dernières positions", fontsize=14, pad=15)
    plt.tight_layout()
    plt.savefig("positions_finales.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("💾 positions_finales.png")
else:
    print("⚠️ Pas assez de joueurs détectés pour le snapshot.")

## ✅ Récapitulatif

In [ ]:
import os
fichiers = ["heatmap_globale.png", "heatmap_ballon.png", "heatmaps_individuelles.png", "positions_finales.png"]
print("📁 Fichiers générés :")
for f in fichiers:
    status = "✅" if os.path.exists(f) else "❌"
    print(f"  {status} {f}")